# IR Assignment 6: PageRank Algorithm (Random Walk vs NetworkX)

In [1]:
%pip install -q networkx

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Assignment 6: PageRank Algorithm using Random Walk
# and NetworkX's built-in PageRank method
import random
import networkx as nx
from collections import Counter

# ------------------------------------------------------------------
# STEP 1: Create a directed web graph
# ------------------------------------------------------------------
G = nx.DiGraph()

# Each edge (A, B) means page A links to page B.
edges = [
    (0, 1), (0, 2), (0, 3),
    (1, 2), (1, 4),
    (2, 0), (2, 3),
    (3, 2), (3, 4),
    (4, 0), (4, 1),
    (5, 2), (5, 4),
    (6, 5), (6, 7),
    (7, 5), (7, 8),
    (8, 6), (8, 9),
    (9, 8), (9, 10),
    (10, 9), (10, 11),
    (11, 10), (11, 12),
    (12, 11), (12, 13),
    (13, 12), (13, 0)
]

G.add_edges_from(edges)

# Ensure every page from 0 to 13 is included
G.add_nodes_from(range(14))

print("Number of web pages:", G.number_of_nodes())
print("Number of links:", G.number_of_edges())

# ------------------------------------------------------------------
# STEP 2: Implement PageRank using Random Walk
# ------------------------------------------------------------------
def random_walk_pagerank(
    graph,
    steps=100000,
    damping_factor=0.85,
    seed=42
):
    random.seed(seed)
    nodes = list(graph.nodes())
    if not nodes:
        return {}

    # Start at a randomly selected page
    current = random.choice(nodes)
    visits = Counter()

    for _ in range(steps):
        visits[current] += 1

        # Teleport with probability (1 - damping_factor)
        if random.random() > damping_factor:
            current = random.choice(nodes)
            continue

        # Follow a random outgoing link
        neighbors = list(graph.successors(current))

        # Handle dead ends: jump to a random page
        if not neighbors:
            current = random.choice(nodes)
        else:
            current = random.choice(neighbors)

    # Convert visit counts into estimated probabilities
    total_visits = sum(visits.values())
    scores = {
        node: visits[node] / total_visits
        for node in nodes
    }
    return scores

# ------------------------------------------------------------------
# STEP 3: Calculate Random Walk PageRank scores
# ------------------------------------------------------------------
random_walk_scores = random_walk_pagerank(
    G,
    steps=100000,
    damping_factor=0.85,
    seed=42
)

print("\nPageRank using Random Walk Method:")
random_walk_ranking = sorted(
    random_walk_scores.items(),
    key=lambda item: item[1],
    reverse=True
)
for page, score in random_walk_ranking:
    print(f"Page {page}: {score:.6f}")

# ------------------------------------------------------------------
# STEP 4: Calculate built-in PageRank scores
# ------------------------------------------------------------------
builtin_scores = nx.pagerank(
    G,
    alpha=0.85,
    max_iter=200,
    tol=1e-10
)

print("\nPageRank using NetworkX built-in method:")
builtin_ranking = sorted(
    builtin_scores.items(),
    key=lambda item: item[1],
    reverse=True
)
for page, score in builtin_ranking:
    print(f"Page {page}: {score:.6f}")

# ------------------------------------------------------------------
# STEP 5: Compare the rankings
# ------------------------------------------------------------------
print("\nComparison of PageRank rankings:")
print(f"{'Page':<10}{'Random Walk':<18}{'Built-in PageRank':<20}")
for page in G.nodes():
    print(
        f"{page:<10}"
        f"{random_walk_scores[page]:<18.6f}"
        f"{builtin_scores[page]:<20.6f}"
    )

# ------------------------------------------------------------------
# STEP 6: Display the most important page
# ------------------------------------------------------------------
most_important_page = max(
    builtin_scores,
    key=builtin_scores.get
)
print("\nMost important page according to built-in PageRank:")
print("Page:", most_important_page)
print("Score:", round(builtin_scores[most_important_page], 6))

# ------------------------------------------------------------------
# STEP 7: Conclusion
# ------------------------------------------------------------------
print("\nConclusion:")
print("PageRank was implemented using the Random Walk method.")
print("The result was compared with NetworkX's built-in PageRank.")

Number of web pages: 14
Number of links: 29

PageRank using Random Walk Method:
Page 2: 0.156650
Page 0: 0.138250
Page 3: 0.116740
Page 4: 0.115490
Page 1: 0.099200
Page 10: 0.057270
Page 11: 0.055930
Page 9: 0.053410
Page 12: 0.048010
Page 8: 0.043430
Page 5: 0.032650
Page 13: 0.030670
Page 6: 0.028980
Page 7: 0.023320

PageRank using NetworkX built-in method:
Page 2: 0.156495
Page 0: 0.139930
Page 3: 0.116871
Page 4: 0.116849
Page 1: 0.100022
Page 10: 0.056495
Page 11: 0.054703
Page 9: 0.053016
Page 12: 0.047007
Page 8: 0.043039
Page 5: 0.032834
Page 13: 0.030692
Page 6: 0.029006
Page 7: 0.023042

Comparison of PageRank rankings:
Page      Random Walk       Built-in PageRank   
0         0.138250          0.139930            
1         0.099200          0.100022            
2         0.156650          0.156495            
3         0.116740          0.116871            
4         0.115490          0.116849            
5         0.032650          0.032834            
6         0.02898